# Static Graph Analysis

Goal: inspect the RPG item graph without running recommendation decoding. Padding item `0` and self-edges are excluded from metrics.


This cell selects the latest completed graph-analysis session and reads metadata such as semantic-ID length.


In [ ]:
from pathlib import Path
import json
import os
import tempfile

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'rpg-matplotlib'))

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

GRAPH_ROOT = REPO_ROOT / 'artifacts' / 'rpg' / 'graph_analysis' / 'sports'
sessions = sorted(p for p in GRAPH_ROOT.iterdir() if (p / 'static' / 'static_summary.csv').is_file())
if not sessions:
    raise FileNotFoundError(f'No completed static graph-analysis sessions found under {GRAPH_ROOT}')
SESSION = sessions[-1]
manifest = json.loads((SESSION / 'manifest.json').read_text())
graph_metadata = json.loads(Path(manifest['graph_metadata']).read_text())
n_digit = int(graph_metadata['n_digit'])
STATIC = SESSION / 'static'

print(SESSION)
print(f"Semantic-ID digits: {n_digit}")
manifest

This cell loads all static-analysis CSVs. `summary` is the compact table; the other files support histograms and per-item plots.


In [ ]:
summary = pd.read_csv(STATIC / 'static_summary.csv')
similarity_hist = pd.read_csv(STATIC / 'edge_similarity_histogram.csv')
hamming_hist = pd.read_csv(STATIC / 'hamming_histogram.csv')
indegree_hist = pd.read_csv(STATIC / 'indegree_histogram.csv')
popularity_buckets = pd.read_csv(STATIC / 'popularity_buckets.csv')
item_metrics = pd.read_csv(STATIC / 'item_metrics_by_k.csv')

summary

## Edge Similarity And Hamming Distance




In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary['k'], summary['edge_similarity_mean'], marker='o', label='graph edges')
ax.plot(summary['k'], summary['random_similarity_mean'], marker='o', label='random pairs')
ax.set_xlabel('raw graph width k, incl. self')
ax.set_ylabel('mean RPG similarity')
ax.set_title('Neighbor similarity vs random pairs')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

a2 = summary.assign(
    matching_digits_mean=n_digit - summary['hamming_mean'],
    random_matching_digits_mean=n_digit - summary['random_hamming_mean'],
)

display(a2[[
    'k',
    'hamming_mean',
    'random_hamming_mean',
    'matching_digits_mean',
    'random_matching_digits_mean',
]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(summary['k'], summary['hamming_mean'], marker='o', label='graph edges')
axes[0].plot(summary['k'], summary['random_hamming_mean'], marker='o', label='random pairs')
axes[0].set_xlabel('raw graph width k, incl. self')
axes[0].set_ylabel(f'mean different digits, max={n_digit}')
axes[0].set_title('Raw Hamming distance, zoomed')
axes[0].set_ylim(max(0, summary['hamming_mean'].min() - 0.5), n_digit + 0.1)
axes[0].legend()

axes[1].plot(a2['k'], a2['matching_digits_mean'], marker='o', label='graph edges')
axes[1].plot(a2['k'], a2['random_matching_digits_mean'], marker='o', label='random pairs')
axes[1].set_xlabel('raw graph width k, incl. self')
axes[1].set_ylabel(f'mean matching digits, out of {n_digit}')
axes[1].set_title('Shared semantic-ID digits')
axes[1].set_ylim(0, max(5, a2['matching_digits_mean'].max() + 0.5))
axes[1].legend()

for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

- `RPG similarity`: embedding similarity used to build the graph. Graph > random means neighbors are semantically meaningful.
- `Hamming distance`: number of semantic-ID digits that differ. Here the maximum is `16`.
- `matching_digits = 16 - hamming`: exact token overlap. `4` means about 4 of 16 semantic-ID digits match.
- Result here: graph neighbors share far more exact digits than random pairs, but they are not near-identical IDs.


## Static Structure

A3-A6 summarize graph shape: reciprocity, connected components, and local clustering.


This cell plots reciprocity, largest-component fraction, and clustering. High clustering vs random means neighborhoods are locally redundant.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(summary['k'], summary['reciprocity'], marker='o')
axes[0].set_title('Reciprocity')
axes[0].set_xlabel('raw graph width k, incl. self')
axes[0].set_ylabel('reciprocal edge fraction')

axes[1].plot(summary['k'], summary['largest_component_fraction'], marker='o')
axes[1].set_title('Largest component')
axes[1].set_xlabel('raw graph width k, incl. self')
axes[1].set_ylabel('node fraction')

axes[2].plot(summary['k'], summary['clustering_mean'], marker='o', label='RPG graph')
axes[2].plot(summary['k'], summary['random_clustering_mean'], marker='o', label='random')
axes[2].set_title('Clustering')
axes[2].set_xlabel('raw graph width k, incl. self')
axes[2].set_ylabel('mean local clustering')
axes[2].legend()

for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
static_structure_table = (
    summary[[
        'k',
        'n_components',
        'largest_component_fraction',
        'clustering_mean',
        'random_clustering_mean',
        'clustering_lift_over_random',
    ]]
    .rename(columns={
        'k': 'raw_graph_width_k_incl_self',
        'largest_component_fraction': 'largest_component_fraction',
        'clustering_mean': 'rpg_clustering',
        'random_clustering_mean': 'random_clustering',
        'clustering_lift_over_random': 'clustering_lift',
    })
    .round({
        'largest_component_fraction': 4,
        'rpg_clustering': 6,
        'random_clustering': 6,
        'clustering_lift': 2,
    })
)

static_structure_table


- Units: these metrics are unitless. They are fractions or ratios, not item counts.
- `Reciprocity`: fraction of directed edges `i -> j` where the reverse edge `j -> i` also exists. High reciprocity means neighborhoods are symmetric.
- `Largest component fraction`: `largest_component_size / n_nodes` after ignoring edge direction. `1.0` means almost all items are in one connected graph.
- `RPG clustering`: average local clustering coefficient. It is the fraction of possible neighbor-neighbor links that actually exist around each item.
- `Random clustering`: the same metric on a random graph with comparable size.
- `Clustering lift`: `RPG clustering / random clustering`; values above `1` mean RPG neighborhoods are more locally redundant than random.
- Result here: the graph is already connected at `k=10`; high clustering/lift is the stronger saturation signal.


## Hubness

checks whether some items receive far more incoming graph edges than expected. Plotting in-degree Gini against a random graph baseline. The histogram shows how many items have each in-degree at the largest `k`.



In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary['k'], summary['indegree_gini'], marker='o', label='graph')
ax.plot(summary['k'], summary['random_indegree_gini_mean'], marker='o', label='random')
ax.set_xlabel('raw graph width k, incl. self')
ax.set_ylabel('in-degree Gini')
ax.set_title('Hubness vs random graph')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

k = int(summary['k'].max())
hist = indegree_hist[indegree_hist['k'] == k]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(hist['indegree'], hist['count'], width=1.0)
ax.set_yscale('log')
ax.set_xlabel('in-degree')
ax.set_ylabel('item count, log scale')
ax.set_title(f'In-degree distribution, k={k}')
plt.show()

- `In-degree`: how often an item appears in other items' neighbor lists.
- `Gini`: inequality of in-degree. `0` is uniform; higher means stronger hubs.
- Histogram: x-axis is in-degree; y-axis is item count on a log scale.
- Result here: hubness exists, but it is not extreme and it decreases as `k` grows.


## Popularity Bias

compares graph in-degree with training frequency to test whether hubs are mostly popular items.


uses the largest `k` slice and plots item popularity against graph in-degree on log scales. The table below reports bucketed in-degree by training-frequency group.


In [ ]:
k = int(summary['k'].max())
items = item_metrics[item_metrics['k'] == k].copy()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    (items['train_frequency'] + 1),
    (items['indegree'] + 1),
    s=8,
    alpha=0.25,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('train frequency + 1, log scale')
ax.set_ylabel('in-degree + 1, log scale')
ax.set_title(f'Popularity vs graph in-degree, k={k}')
ax.grid(alpha=0.25)
plt.show()

popularity_buckets[popularity_buckets['k'] == k]

- `train_frequency`: how often an item appears in the training split.
- `in-degree`: how often the item is selected as a graph neighbor.
- Scatter: positive slope would mean popular items become graph hubs.
- Bucket table: compares in-degree across popularity groups.
- Result here: popularity relation is positive but weak, so popularity is not the main static explanation.
